In [ ]:
import pandas as pd
import numpy as np
import gensim.downloader as api
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

glove = api.load("glove-wiki-gigaword-100")  # 100-dim GloVe

TRAINING_DATA_PATH = "training_data/ED/train.csv"
DEV_DATA_PATH = "training_data/ED/dev.csv"

training_data_df = pd.read_csv(TRAINING_DATA_PATH)


KeyedVectors<vector_size=100, 400000 keys>


In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

def tokens_to_vectors(tokens, glove, dim=100):
    vecs = [glove[t] if t in glove else np.zeros(dim) for t in tokens]
    return np.array(vecs)

training_data_df["claim_tokens"] = training_data_df["Claim"].apply(preprocess)
training_data_df["evidence_tokens"] = training_data_df["Evidence"].apply(preprocess)

training_data_df["claim_vecs"] = training_data_df["claim_tokens"].apply(lambda t: tokens_to_vectors(t, glove))
training_data_df["evidence_vecs"] = training_data_df["evidence_tokens"].apply(lambda t: tokens_to_vectors(t, glove))


In [ ]:
from torch.nn.utils.rnn import pad_sequence

class ClaimEvidenceDataSet(Dataset):
    def __init__(self, claim, evidence, label=None):
        self.claim = [torch.tensor(c, dtype=torch.float32) for c in claim]
        self.evidence = [torch.tensor(e, dtype=torch.float32) for e in evidence]
        self.label = torch.tensor(label.values, dtype=torch.float32) if label is not None else None

    def __len__(self):
        return len(self.claim)

    def __getitem__(self, idx):
        item = {"claim": self.claim[idx], "evidence": self.evidence[idx]}
        label = self.label[idx] if self.label is not None else None
        return item, label

def collate_fn(batch):
    items, labels = zip(*batch)  # always a tuple now
    claims = pad_sequence([x["claim"] for x in items], batch_first=True)
    evidences = pad_sequence([x["evidence"] for x in items], batch_first=True)
    result = {"claim": claims, "evidence": evidences}
    if labels[0] is not None:
        return result, torch.stack(labels)
    return result, None


class LSTMEvidenceDetectionModel(nn.Module):

    def __init__(self, 
                 embedding_size,
                 num_layers,
                 p_dropout=0.2               
                 ):
        super().__init__()
        self.claim_lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=embedding_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=p_dropout
        )

        self.evidence_lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=embedding_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=p_dropout
        )

        self.classification_head = nn.Sequential(
            nn.LayerNorm(4 * embedding_size),
            nn.Linear(4 * embedding_size, embedding_size),
            nn.Dropout(p_dropout),
            nn.GELU(),
            nn.Linear(embedding_size, 1)
        )

    def forward(self, X):
        claim = X["claim"]
        evidence = X["evidence"]

        claim_out, _ = self.claim_lstm(claim)
        evidence_out, _ = self.evidence_lstm(evidence)

        claim_hidden = claim_out[:, -1, :]
        evidence_hidden = evidence_out[:, -1, :]

        combined = torch.cat((claim_hidden, evidence_hidden), dim=-1)
        return self.classification_head(combined)


In [ ]:
def train_step(model, loss_fn, optimiser, device, X, y):
    model.train()
    outputs = model(X)
    loss = loss_fn(outputs.squeeze(-1), y)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    return {"loss": loss.item()}

@torch.no_grad()
def test_step(model, loss_fn, device, X, y):
    model.eval()
    outputs = model(X)
    loss = loss_fn(outputs.squeeze(-1), y)
    return {"loss": loss.item()}


def train_loop(model,
               loss_fn,
               optimiser,
               training_dataloader,
               validation_dataloader=None,
               device="cuda" if torch.cuda.is_available() else "cpu",
               n_epochs=10):

    model.to(device)
    train_loss = []

    for epoch in range(n_epochs):
        train_epoch_loss = 0
        for X, y in training_dataloader:
            X = {k: v.to(device) for k, v in X.items()}  # move dict tensors to device
            y = y.to(device)
            outputs = train_step(model, loss_fn, optimiser, device, X, y)
            train_epoch_loss += outputs["loss"]

        train_epoch_loss /= len(training_dataloader)
        train_loss.append(train_epoch_loss)
        print(f"Epoch {epoch+1}/{n_epochs} — loss: {train_epoch_loss:.4f}")

        if validation_dataloader:
            val_epoch_loss = 0
            for X, y in validation_dataloader:
                X = {k: v.to(device) for k, v in X.items()}
                y = y.to(device)
                outputs = test_step(model, loss_fn, device, X, y)
                val_epoch_loss += outputs["loss"]
            val_epoch_loss /= len(validation_dataloader)
            print(f"  val loss: {val_epoch_loss:.4f}")



In [ ]:
model = LSTMEvidenceDetectionModel(100, 5)

training_dataset = ClaimEvidenceDataSet(training_data_df["claim_vecs"], training_data_df["evidence_vecs"], training_data_df["label"])
training_dataloader = DataLoader(training_dataset, 64 , collate_fn = collate_fn)

loss_function = nn.BCEWithLogitsLoss()
optimiser = torch.optim.Adam(model.parameters(), lr = 1e-3)


train_loop(
    model,
    loss_function,
    optimiser,
    training_dataloader,
    n_epochs = 50
)

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

dev_data_df = pd.read_csv(DEV_DATA_PATH)

dev_data_df["claim_tokens"] = dev_data_df["Claim"].apply(preprocess)
dev_data_df["evidence_tokens"] = dev_data_df["Evidence"].apply(preprocess)
dev_data_df["claim_vecs"] = dev_data_df["claim_tokens"].apply(lambda t: tokens_to_vectors(t, glove))
dev_data_df["evidence_vecs"] = dev_data_df["evidence_tokens"].apply(lambda t: tokens_to_vectors(t, glove))

dev_dataset = ClaimEvidenceDataSet(dev_data_df["claim_vecs"], dev_data_df["evidence_vecs"])
dev_dataloader = DataLoader(dev_dataset, batch_size=64, collate_fn=collate_fn)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
all_probs = []

with torch.no_grad():
    for X, _ in dev_dataloader:
        X = {k: v.to(device) for k, v in X.items()}
        logits = model(X).squeeze(-1)
        all_probs.extend(torch.sigmoid(logits).cpu().tolist())

all_preds = (np.array(all_probs) >= 0.5).astype(int)
dev_data_df["predicted_label"] = all_preds

y_true = dev_data_df["label"].values
y_pred = dev_data_df["predicted_label"].values

accuracy = (y_pred == y_true).mean()
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_binary = f1_score(y_true, y_pred, average="binary")

print(f"Dev accuracy:  {accuracy:.4f}")
print(f"F1 (binary):   {f1_binary:.4f}")
print(f"F1 (macro):    {f1_macro:.4f}")
print("\nFull report:")
print(classification_report(y_true, y_pred))


In [ ]:
# Threshold sweep — reuse all_probs from above cell
thresholds = np.arange(0.1, 1.0, 0.05)
y_true = dev_data_df["label"].values

results = []
for t in thresholds:
    preds = (np.array(all_probs) >= t).astype(int)
    f1 = f1_score(y_true, preds, average="binary", zero_division=0)
    acc = (preds == y_true).mean()
    results.append({"threshold": round(t, 2), "f1_binary": round(f1, 4), "accuracy": round(acc, 4)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

best = results_df.loc[results_df["f1_binary"].idxmax()]
print(f"\nBest threshold: {best['threshold']}  →  F1: {best['f1_binary']}  Accuracy: {best['accuracy']}")
